# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS42024/shailesh-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# One row = one page in one calendar month which is March 2026. Each Row Summarizes how that page performed in the search of that Mont. I Chose a mid panel month so that June 2026, the final month, stays sealed as a Test Month.

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
df = pd.DataFrame({
    "clicks_last_month": rng.poisson(20, 2000),
    "position": rng.uniform(1, 30, 2000),      # was "positions"
})
df["label"] = (df["clicks_last_month"] + rng.normal(0, 15, 2000) > 25).astype(int)

def score(features):
    X_tr, X_te, y_tr, y_te = train_test_split(
        df[features], df["label"], test_size=0.3, random_state=0)
    m = RandomForestClassifier(random_state=0).fit(X_tr, y_tr)
    return roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])

print("honest:", score(["clicks_last_month", "position"]))

df["leak"] = df["label"] + rng.normal(0, 0.05, 2000)
print("with leak:", score(["clicks_last_month", "position", "leak"]))

honest: 0.5586962667945078
with leak: 1.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
!pip -q install duckdb huggingface_hub
import duckdb
from getpass import getpass
from huggingface_hub import HfApi

# 1) Get the token: Colab Secret first, hidden prompt as a fallback
def get_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception as e:
        print(f"Secret 'HF_TOKEN' not usable ({type(e).__name__}). "
              "Add it via the key icon and turn on Notebook access. "
              "For now, paste it below (hidden, not saved).")
        return getpass("HF READ token: ")

token = get_token()

# 2) Check the token itself is valid
try:
    print("Token OK for user:", HfApi().whoami(token=token)["name"])
except Exception:
    raise SystemExit("That token was rejected. Create a new plain READ token in HF settings.")

# 3) Connect DuckDB, then drop the token from memory
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")
del token

# 4) Find a file path that works (tries the partition folder, then a wildcard)
REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

def month_src(m):
    options = [
        f"read_parquet('{REL}/month={m}/*.parquet')",
        f"(SELECT * FROM read_parquet('{REL}/**/*.parquet', hive_partitioning=true) WHERE month = '{m}')",
    ]
    for src in options:
        try:
            con.sql(f"SELECT 1 FROM {src} LIMIT 1").fetchall()
            return src
        except Exception as e:
            last = str(e)
    if "403" in last or "401" in last:
        raise SystemExit("Access denied: request access on the dataset page and use a plain READ token.")
    raise SystemExit(f"No working path found. Last error:\n{last}")

MARCH, APRIL = "2026-03", "2026-04"
src = month_src(MARCH)

cols = con.sql(f"DESCRIBE SELECT * FROM {src}").df()[["column_name", "column_type"]]
print(cols.to_string())

Secret 'HF_TOKEN' not usable (SecretNotFoundError). Add it via the key icon and turn on Notebook access. For now, paste it below (hidden, not saved).


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:

#Grain. One row per `<KEY columns>`. The query below shows `<N>` rows and `<N>` distinct keys, with `<0>` duplicate keys, so the claim holds.

#Counts. The dataset has `<N>` rows, which matches the contract's expected size.

#Missing values. The most incomplete column is `<column>` at `<X>%` missing. All fields planned as features have less than `<Y>%` missing, so they are safe to use. `<Column>` is excluded because `<X>%` of its values are missing.

#Window. The data covers `<min date>` to `<max date>`, matching the contract's stated window. `<No months are missing / Month M has no rows>`.


from itertools import combinations

print("rows:", len(df))
print("fully duplicated rows:", df.duplicated().sum())
print(df.nunique().sort_values(ascending=False).head(10))

# Search for the smallest set of columns that uniquely identifies a row
cols_ok = [c for c in df.columns if df[c].isna().mean() < 0.05]
KEY = None
for size in (1, 2, 3):
    for cols in combinations(cols_ok, size):
        if not df.duplicated(subset=list(cols)).any():
            KEY = list(cols)
            break
    if KEY:
        break

if KEY:
    print("\nGRAIN FOUND. One row per:", KEY)
    print("rows:", len(df), "| distinct keys:", df[KEY].drop_duplicates().shape[0])
else:
    # No unique key exists, so report the closest one and how many rows repeat
    best = df.nunique().sort_values(ascending=False).index[0]
    dupes = df.duplicated(subset=[best], keep=False).sum()
    print("\nNO UNIQUE KEY. Closest column:", best)
    print(f"rows sharing a {best} value with another row:", dupes)
    KEY = [best]
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


rows: 17000
fully duplicated rows: 0
median_income         11175
total_rooms            5533
median_house_value     3694
population             3683
total_bedrooms         1848
households             1740
latitude                840
longitude               827
housing_median_age       52
dtype: int64

GRAIN FOUND. One row per: ['longitude', 'total_rooms', 'total_bedrooms']
rows: 17000 | distinct keys: 17000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [26]:
# Unbalanced history: rows per month
if DATE_COL:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    monthly = df.groupby(df[DATE_COL].dt.to_period("M")).size()
    print("rows per month:\n", monthly)
    print("min month:", monthly.idxmin(), monthly.min(), "| max month:", monthly.idxmax(), monthly.max())

# GSC-only early rows: does missingness change over time?
    early = df[df[DATE_COL] <= df[DATE_COL].min() + pd.Timedelta(days=30)]
    later = df[df[DATE_COL] > df[DATE_COL].min() + pd.Timedelta(days=30)]
    print("\nmissing share, first 30 days:\n", early.isna().mean().sort_values(ascending=False).head())
    print("\nmissing share, rest of data:\n", later.isna().mean().sort_values(ascending=False).head())

# Window overlaps: if you have two date-bounded sources/files, compare their ranges
    print("\noverall min/max date:", df[DATE_COL].min(), df[DATE_COL].max())
else:
    print("No DATE_COL set — skip these checks or set DATE_COL to your date column first.")


# **Data limits**


 #Unbalanced history.** Not every page has the same amount of history in this dataset. Newer pages have fewer months of data than older ones, so a page with a short history looks more volatile just because there's less data to average out, not because it actually performs less consistently.

 #GSC-only early rows.** The earliest records in this data come only from Google Search Console metrics (impressions, clicks, position), before other signals were being tracked. This means trends that reach back into that early period are comparing a narrower set of metrics than trends from later periods, so they aren't a fair apples-to-apples comparison.

 #Window overlaps.** This sample is a fixed window pulled from a larger warehouse. Rows near the edges of that window may be undercounted, since activity just before or after the cutoff isn't fully captured, and if this sample is ever combined with another cut of the same warehouse, overlapping rows could get double-counted.

#What this means for modeling:** conclusions drawn near the start of the data's history, or near the edges of the sampled window, are less reliable than conclusions from the stable middle of the range. The model shouldn't be expected to explain performance changes it never saw a complete, unbiased picture of.

No DATE_COL set — skip these checks or set DATE_COL to your date column first.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.